# Daily Briefing System — MCP-Powered News Agents

A **Personal Daily Briefing System** that blends every concept from Week 6 into one project:

| Week 6 Concept | How It Appears Here |
|---|---|
| **Lab 1** — Using existing MCP servers | Serper Search + Fetch for news research |
| **Lab 2** — Custom MCP server + client + resources | `briefing_server.py` with tools (`save_article`, `get_articles`) and resources (`briefings://today/{reporter}`, `briefings://beat/{reporter}`) |
| **Lab 3** — Memory, Serper Search, 3 server types | Knowledge-graph memory per reporter, Serper for web search, custom local server |
| **Lab 4** — Researcher sub-agent as tool, MCP resources | Fact-checking Researcher agent wrapped as a tool via `.as_tool()`, beat info read from MCP resources |
| **Lab 5** — Multi-agent loop, tracing, Gradio UI | 3 reporters run concurrently, custom `BriefingTracer`, live dashboard |

**Three reporters**, each covering a different beat:
- **Ada** — Tech & AI
- **Marcus** — Finance & Markets
- **Zara** — Science & Innovation

Each reporter has a **Researcher sub-agent** (wrapped as a tool) that uses Fetch + Serper Search + Memory to investigate stories.

In [1]:
import os
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown
from datetime import datetime

load_dotenv(override=True)

openai_key = os.getenv("OPENAI_API_KEY")
serper_key = os.getenv("SERPER_API_KEY")

print(f"OpenAI Key: {openai_key[:8]}..." if openai_key else "OPENAI_API_KEY not set")
print(f"Serper Key: {serper_key[:8]}..." if serper_key else "SERPER_API_KEY not set — web search will not work")

OpenAI Key: sk-proj-...
Serper Key: 1b0d07d2...


## Step 1 — Explore Existing MCP Servers (Lab 1 & 3)

Before building anything custom, let's confirm the MCP servers we'll use work.
Serper provides Google-powered web search + page scraping via an MCP server,
and Fetch retrieves full web pages via a headless browser.

In [2]:
serper_env = {"SERPER_API_KEY": os.getenv("SERPER_API_KEY")}
serper_params = {"command": "npx", "args": ["-y", "serper-search-scrape-mcp-server"], "env": serper_env}

async with MCPServerStdio(params=serper_params, client_session_timeout_seconds=60) as server:
    serper_tools = await server.list_tools()

print(f"Serper Search: {len(serper_tools)} tools")
for t in serper_tools:
    print(f"  - {t.name}: {t.description[:80]}...")

Serper Search: 2 tools
  - google_search: Tool to perform web searches via Serper API and retrieve rich results. It is abl...
  - scrape: Tool to scrape a webpage and retrieve the text and, optionally, the markdown con...


In [3]:
fetch_params = {"command": "uvx", "args": ["mcp-server-fetch"]}

async with MCPServerStdio(params=fetch_params, client_session_timeout_seconds=60) as server:
    fetch_tools = await server.list_tools()

print(f"Fetch: {len(fetch_tools)} tools")
for t in fetch_tools:
    print(f"  - {t.name}: {t.description[:80]}...")

Fetch: 1 tools
  - fetch: Fetches a URL from the internet and optionally extracts its contents as markdown...


## Step 2 — Build Our Custom MCP Server (Lab 2)

The `briefing_server.py` module is our custom MCP server, built with Anthropic's `FastMCP`.
It provides:

**Tools:**
- `save_article` — file a news article to the database
- `get_articles` — retrieve recent articles by reporter
- `update_beat` — change a reporter's beat assignment

**Resources:**
- `briefings://today/{reporter}` — today's articles for a reporter
- `briefings://beat/{reporter}` — the reporter's beat description

Backed by SQLite via `briefing_database.py` — the same pattern as the course's `accounts_server.py` + `database.py`.

First, let's initialize our three reporters.

In [4]:
from briefing_server import reset_reporters
reset_reporters()

Reporters reset: Ada (Tech & AI), Marcus (Finance & Markets), Zara (Science & Innovation)


In [5]:
briefing_params = {"command": "uv", "args": ["run", "briefing_server.py"]}

async with MCPServerStdio(params=briefing_params, client_session_timeout_seconds=30) as server:
    briefing_tools = await server.list_tools()

print(f"Briefing Server: {len(briefing_tools)} tools")
for t in briefing_tools:
    print(f"  - {t.name}: {t.description[:80]}...")

Briefing Server: 3 tools
  - save_article: Save a news article to the briefing database.

    Args:
        reporter: The n...
  - get_articles: Get recent articles filed by a specific reporter.

    Args:
        reporter: T...
  - update_beat: Update a reporter's beat assignment and focus description.

    Args:
        re...


### Test the custom MCP server with an agent

Let's give an agent access to our briefing server and ask it to save a test article.

In [6]:
instructions = "You can save and retrieve news articles using your tools."
request = "Save a test article under reporter 'ada' with headline 'MCP Protocol Gains Momentum', summary 'The Model Context Protocol continues to see rapid adoption across AI agent frameworks.', and sources 'anthropic.com'"

async with MCPServerStdio(params=briefing_params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="tester", instructions=instructions, model="gpt-4o-mini", mcp_servers=[mcp_server])
    with trace("test-briefing-server"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

The article titled "MCP Protocol Gains Momentum" has been successfully saved under reporter 'ada'.

### Test MCP Resources via our client (Lab 2)

The MCP client helpers in `reporters.py` connect to the MCP server and read resources —
the same pattern as `accounts_client.py` from the course.

In [7]:
from reporters import read_todays_briefing, read_beat_resource

beat = await read_beat_resource("ada")
print("Ada's beat:")
print(beat)

print("\n---\n")

articles = await read_todays_briefing("ada")
print("Ada's recent articles:")
print(articles)

Ada's beat:
Beat: Tech & AI
Description: 
You are Ada, covering Technology & AI.
You focus on artificial intelligence breakthroughs, major software product launches,
cybersecurity incidents, and big tech industry moves. You look for stories that signal
where technology is heading — new model releases, startup funding rounds, regulatory changes,
and innovative applications of AI in the real world.
You have a knack for explaining complex technical topics in accessible language.


---

Ada's recent articles:
[
  {
    "headline": "MCP Protocol Gains Momentum",
    "summary": "The Model Context Protocol continues to see rapid adoption across AI agent frameworks.",
    "sources": "anthropic.com",
    "datetime": "2026-03-30 10:18:10"
  }
]


## Step 3 — Knowledge Graph Memory (Lab 3)

Each reporter gets a persistent knowledge graph via `mcp-memory-libsql`, stored
in a per-reporter SQLite database under `memory/`. This lets reporters build
expertise over time and avoid repeating stories.

In [8]:
memory_params = {"command": "npx", "args": ["-y", "mcp-memory-libsql"], "env": {"LIBSQL_URL": "file:./memory/test.db"}}

async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=30) as server:
    memory_tools = await server.list_tools()

print(f"Memory: {len(memory_tools)} tools")
for t in memory_tools:
    print(f"  - {t.name}")

Memory: 6 tools
  - create_entities
  - search_nodes
  - read_graph
  - create_relations
  - delete_entity
  - delete_relation


In [9]:
instructions = "You use your entity tools as persistent memory to store and recall information."
request = "Remember this: Ada is a tech reporter who covers AI and cybersecurity. She recently wrote about MCP adoption."

async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="memory_test", instructions=instructions, model="gpt-4o-mini", mcp_servers=[mcp_server])
    with trace("memory-test"):
        result = await Runner.run(agent, request)
    print(result.final_output)

print("\n--- Now recalling ---\n")

async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="memory_test", instructions=instructions, model="gpt-4o-mini", mcp_servers=[mcp_server])
    with trace("memory-recall"):
        result = await Runner.run(agent, "What do you know about Ada?")
    print(result.final_output)

I've noted that Ada is a tech reporter covering AI and cybersecurity, and she recently wrote about MCP adoption. Let me know if there's anything else you'd like to add!

--- Now recalling ---

I found an entity named Ada. Here are some details:

- **Type**: Person
- **Observations**:
  - Tech reporter
  - Covers AI and cybersecurity
  - Recently wrote about MCP adoption

If you need more specific information or have any particular queries about Ada, let me know!


## Step 4 — Researcher Sub-Agent as Tool (Lab 4)

The Researcher agent uses Fetch + Serper Search + Memory to investigate stories.
We wrap it as a tool with `.as_tool()` so reporters can delegate research.

This is exactly the pattern from the trading floor — the Researcher agent
is the equivalent of the Trader's research tool.

In [10]:
from reporters import researcher_mcp_server_params

researcher_params = researcher_mcp_server_params("ada")
researcher_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in researcher_params]

for server in researcher_mcp_servers:
    await server.connect()

researcher = Agent(
    name="Researcher",
    instructions=f"""You are a news researcher. Search the web for current news,
verify facts, and produce concise summaries. Make multiple searches for comprehensive coverage.
The current datetime is {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}""",
    model="gpt-4o-mini",
    mcp_servers=researcher_mcp_servers,
)

with trace("researcher-test"):
    result = await Runner.run(researcher, "What are the biggest AI news stories today?", max_turns=30)

display(Markdown(result.final_output))

Here are some of the biggest AI news stories today:

1. **Tech CEOs Blaming AI for Job Cuts**  
   More tech leaders are attributing job losses to the rise of AI tools, highlighting a demand for increased investment in this area. [Read more](https://www.bbc.com/news/articles/cde5y2x51y8o) (10 hours ago)

2. **Pro-AI Campaign by Trump Allies**  
   Allies of Donald Trump have launched a new $100 million campaign aiming to influence tech policy in the upcoming 2026 midterms, focusing on AI advocacy. [Read more](https://www.foxnews.com/politics/new-pro-ai-group-backed-trump-allies-plans-100m-midterm-spending-push) (16 hours ago)

3. **AI in Drone Warfare**  
   The architect of Ukraine's drone program discussed how AI-powered drone swarms could provide a significant advantage in warfare, showcasing the future implications of AI in military applications. [Read more](https://www.cbsnews.com/news/drone-swarms-the-potential-ai-future-of-drone-warfare-60-minutes/) (10 hours ago)

4. **AI Leading to Job Losses**  
   A former employee at PwC warns that workers not utilizing AI face significant risks as companies increasingly adopt these technologies. [Read more](https://www.abc.net.au/news/2026-03-30/ai-jobs-cuts-companies-losses-agents-replacement-humans/106491600) (3 hours ago)

5. **Senator Mark Warner's Thoughts on AI Risks**  
   Senator Mark Warner expressed concern about the risks associated with AI, stating he wants to remain optimistic but feels "terrified" about its implications. [Read more](https://www.reddit.com/r/news/comments/1s72wts/senator_mark_warner_on_ais_risks_i_want_to_be/) 

These stories reflect the growing influence and concerns surrounding AI in technology, politics, and society.

In [11]:
researcher_tool = researcher.as_tool(
    tool_name="Researcher",
    tool_description="Researches the web for current news. Describe what topic or story to investigate."
)

print(f"Researcher wrapped as tool: {researcher_tool.name}")
print(f"Description: {researcher_tool.description}")

Researcher wrapped as tool: Researcher
Description: Researches the web for current news. Describe what topic or story to investigate.


### Check the trace

https://platform.openai.com/traces

You should see the Researcher calling Serper Search and Fetch tools to investigate stories.

## Step 5 — Build a Single Reporter (Lab 4)

Now we wire everything together into a full reporter agent:
- **Researcher** as a tool (for investigating stories)
- **Briefing Server** MCP (for saving/reading articles)
- **Beat info** read from MCP resource (injected into the prompt)

This mirrors the Trader from Lab 4 — the Trader used accounts + market + push MCP servers
and a Researcher tool, while our Reporter uses briefing + memory MCP servers and a Researcher tool.

In [12]:
from reporters import read_beat_resource, read_todays_briefing
from reporters import briefing_mcp_server_params, researcher_mcp_server_params
from reporters import reporter_instructions, briefing_message

agent_name = "Ada"

beat = await read_beat_resource(agent_name)
previous_articles = await read_todays_briefing(agent_name)

print("Beat:")
print(beat)
print("\nPrevious articles:")
print(previous_articles)

Beat:
Beat: Tech & AI
Description: 
You are Ada, covering Technology & AI.
You focus on artificial intelligence breakthroughs, major software product launches,
cybersecurity incidents, and big tech industry moves. You look for stories that signal
where technology is heading — new model releases, startup funding rounds, regulatory changes,
and innovative applications of AI in the real world.
You have a knack for explaining complex technical topics in accessible language.


Previous articles:
[
  {
    "headline": "MCP Protocol Gains Momentum",
    "summary": "The Model Context Protocol continues to see rapid adoption across AI agent frameworks.",
    "sources": "anthropic.com",
    "datetime": "2026-03-30 10:18:10"
  }
]


In [13]:
briefing_servers = [MCPServerStdio(p, client_session_timeout_seconds=60) for p in briefing_mcp_server_params]
research_servers = [MCPServerStdio(p, client_session_timeout_seconds=60) for p in researcher_mcp_server_params(agent_name)]
all_servers = briefing_servers + research_servers

for server in all_servers:
    await server.connect()

researcher_agent = Agent(
    name="Researcher",
    instructions=f"""You are a news researcher. Search the web for current news and produce factual summaries.
The current datetime is {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}""",
    model="gpt-4o-mini",
    mcp_servers=research_servers,
)
research_tool = researcher_agent.as_tool(
    tool_name="Researcher",
    tool_description="Researches the web for current news. Describe what to investigate.",
)

reporter = Agent(
    name=agent_name,
    instructions=reporter_instructions(agent_name),
    model="gpt-4o-mini",
    tools=[research_tool],
    mcp_servers=briefing_servers,
)

message = briefing_message(agent_name, beat, previous_articles)

with trace(f"{agent_name}-briefing"):
    result = await Runner.run(reporter, message, max_turns=30)

display(Markdown(result.final_output))

Today, I filed three articles covering significant developments in technology and AI:

1. **SoftBank Plans $40 Billion Investment in OpenAI**: SoftBank is set to invest heavily in OpenAI, reflecting a bold move in the evolving landscape of AI technologies.

2. **California Advances Regulatory Measures for Big Tech**: The state is stepping up its regulatory efforts to ensure better data privacy and ethical practices among major technology companies.

3. **OpenAI Launches GPT-5.4 Model**: OpenAI has unveiled its GPT-5.4 model, featuring advanced capabilities such as a 1-million-token context window, enhancing its accessibility for developers.

These stories signal ongoing changes and advancements in both the tech and AI sectors.

In [14]:
updated_articles = await read_todays_briefing("Ada")
print("Ada's articles after running:")
display(Markdown(updated_articles))

Ada's articles after running:


[
  {
    "headline": "SoftBank Plans $40 Billion Investment in OpenAI",
    "summary": "SoftBank aims to invest $40 billion into OpenAI, signaling a significant bet on the future of AI technologies and a reevaluation of industry dynamics.",
    "sources": "techstartups.com",
    "datetime": "2026-03-30 10:21:01"
  },
  {
    "headline": "California Advances Regulatory Measures for Big Tech",
    "summary": "California is increasing efforts to regulate Big Tech companies, focusing on enhancing data privacy and ethical practices in technology usage.",
    "sources": "cnn.com",
    "datetime": "2026-03-30 10:21:01"
  },
  {
    "headline": "OpenAI Launches GPT-5.4 Model",
    "summary": "OpenAI has introduced its new GPT-5.4 model, which boasts advanced features including a 1-million-token context window, enhancing its capabilities for developers.",
    "sources": "champaignmagazine.com",
    "datetime": "2026-03-30 10:21:01"
  },
  {
    "headline": "MCP Protocol Gains Momentum",
    "summary": "The Model Context Protocol continues to see rapid adoption across AI agent frameworks.",
    "sources": "anthropic.com",
    "datetime": "2026-03-30 10:18:10"
  }
]

### Check the trace

https://platform.openai.com/traces

You should see Ada calling the Researcher tool (which uses Serper Search + Fetch),
then calling `save_article` on the Briefing Server to file stories.

## Step 6 — Review the Python Modules

Just as the course migrated from notebooks to Python modules, we've built:

| Module | Purpose | Course Equivalent |
|---|---|---|
| `briefing_database.py` | SQLite persistence (reporters, articles, logs) | `database.py` |
| `briefing_server.py` | Custom MCP server with tools + resources; reporter initialization | `accounts_server.py` + `reset.py` |
| `reporters.py` | Reporter agent class, Researcher sub-agent, MCP client, params, templates, tracing, briefing floor | `traders.py` + `accounts_client.py` + `mcp_params.py` + `templates.py` + `tracers.py` + `trading_floor.py` |
| `briefing_app.py` | Gradio dashboard with 3 reporter columns | `app.py` |

The `AsyncExitStack` pattern from Lab 4 is used in `reporters.py` to cleanly
manage multiple MCP server connections without deeply nested `async with` blocks.

## Step 7 — Scale to 3 Reporters (Lab 5)

Now we use the `Reporter` class from `reporters.py` to run all three reporters.
This mirrors `trading_floor.py` — all reporters run concurrently via `asyncio.gather`.

In [15]:
from reporters import Reporter

ada = Reporter("Ada")
marcus = Reporter("Marcus")
zara = Reporter("Zara")

print("Running all 3 reporters concurrently...")
print("This will take a few minutes as each reporter researches and files stories.\n")

import asyncio
await asyncio.gather(ada.run(), marcus.run(), zara.run())

print("\nAll reporters have filed their briefings!")

Running all 3 reporters concurrently...
This will take a few minutes as each reporter researches and files stories.


All reporters have filed their briefings!


In [16]:
from briefing_database import read_all_articles_today

all_articles = read_all_articles_today()
print(f"Total articles filed today: {len(all_articles)}\n")

for article in all_articles:
    print(f"[{article['reporter'].upper()}] {article['headline']}")
    print(f"  {article['summary'][:120]}...")
    print(f"  Sources: {article['sources']}")
    print()

Total articles filed today: 13

[ZARA] Scientists Successfully Transport Antimatter for Stability Assessment
  For the first time, researchers have transported antimatter particles by road to evaluate their stability during transit...
  Sources: https://en.wikipedia.org/wiki/2026_in_science

[ZARA] NASA Prepares for Artemis II Mission Around the Moon
  NASA's Artemis II mission is gearing up to be the first U.S. spacecraft in nearly 54 years to circumnavigate the Moon. A...
  Sources: https://www.cnn.com/2026/03/29/us/video/artemis-ii-is-set-to-be-the-first-u-s-spacecraft-in-nearly-54-years-to-fly-around-the-moon-and-return-to-earth

[ZARA] Beavers Identified as Natural Carbon Emission Mitigators
  Research suggests that beavers could serve as a natural solution for reducing carbon emissions, highlighting the importa...
  Sources: https://www.livescience.com/news/archive/2026/03

[MARCUS] Stock Market Faces Significant Declines Amid Geopolitical Tensions
  As of March 27, 2026, the sto

## Step 8 — Custom Tracing (Lab 5)

The `BriefingTracer` class in `reporters.py` extends OpenAI's `TracingProcessor` to intercept trace events
and write them to our SQLite `logs` table — the same pattern as `tracers.py` in the course.

This feeds the live activity log in the Gradio dashboard.

In [17]:
from briefing_database import read_log

for name in ["ada", "marcus", "zara"]:
    logs = read_log(name, last_n=5)
    if logs:
        print(f"\n--- {name.upper()} recent activity ---")
        for timestamp, log_type, message in logs:
            print(f"  [{log_type}] {message}")

## Step 9 — Gradio Dashboard & Briefing Floor (Lab 5)

Launch the Gradio dashboard directly from the notebook, then run
a single briefing cycle to see the reporters populate it in real-time.

In [ ]:
from briefing_app import create_ui

ui = create_ui()
ui.launch(inbrowser=True)

### Run one briefing cycle

This kicks off all 3 reporters concurrently — the same thing `briefing_floor.py` does
in its `while True` loop, but just a single pass. Watch the dashboard update as they file stories.

In [ ]:
from reporters import Reporter, BriefingTracer
from agents import add_trace_processor
import asyncio

add_trace_processor(BriefingTracer())

reporters = [Reporter("Ada"), Reporter("Marcus"), Reporter("Zara")]

print("Running all 3 reporters concurrently with tracing enabled...")
print("Watch the Gradio dashboard update in real-time!\n")

await asyncio.gather(*[r.run() for r in reporters])

print("\nBriefing cycle complete!")

### For continuous running

To run the briefing floor on a loop (like `trading_floor.py`), open a terminal, `cd` to this directory, and run:

```
uv run reporters.py
```

This runs the same cycle above every N minutes (default 60). Set `RUN_EVERY_N_MINUTES=30` in your `.env` to change the interval.

## Step 10 — Count Our MCP Servers & Tools

Let's see the full scope of what we've built.

In [18]:
from reporters import briefing_mcp_server_params, researcher_mcp_server_params

all_params = briefing_mcp_server_params + researcher_mcp_server_params("ada")

count = 0
for each_params in all_params:
    async with MCPServerStdio(params=each_params, client_session_timeout_seconds=60) as server:
        mcp_tools = await server.list_tools()
        count += len(mcp_tools)

print(f"We have {len(all_params)} MCP servers and {count} tools per reporter")
print(f"Across 3 reporters: {len(all_params) * 3} total MCP server instances")

We have 4 MCP servers and 12 tools per reporter
Across 3 reporters: 12 total MCP server instances


## Review — What We Built

This project demonstrates every MCP concept from Week 6:

**Custom MCP Server** (`briefing_server.py`)
- 3 tools: `save_article`, `get_articles`, `update_beat`
- 2 resources: `briefings://today/{reporter}`, `briefings://beat/{reporter}`
- Reporter initialization with beat descriptions
- Backed by SQLite, exposed via `FastMCP` with Stdio transport

**MCP Client & Agent Infrastructure** (`reporters.py`)
- Reads MCP resources to inject reporter context into prompts
- Converts MCP tools to OpenAI `FunctionTool` objects
- MCP server parameters for all servers (briefing, fetch, serper, memory)
- Prompt templates for reporters and researchers
- Custom `BriefingTracer` for live activity logging
- Concurrent execution loop via `asyncio.gather`

**Existing MCP Servers**
- Serper Search — Google-powered web search + page scraping
- Fetch — full page retrieval via headless browser
- Memory (libsql) — persistent knowledge graph per reporter

**Agent Architecture**
- Researcher sub-agent wrapped as tool via `.as_tool()`
- Reporter agents with MCP servers for briefing + memory
- `AsyncExitStack` for clean multi-server lifecycle management

### Architecture

```
reporters.py (main loop, every N minutes)
  ├── Reporter("Ada")    ──┐
  ├── Reporter("Marcus")  ──┤  all run concurrently via asyncio.gather
  └── Reporter("Zara")   ──┘
        │
        ├── MCP: briefing_server (save_article, get_articles)
        ├── Researcher Sub-Agent (as tool)
        │     └── MCP: fetch, serper_search, memory (per-reporter DB)
        └── Beat info read via MCP resource

briefing_app.py (Gradio dashboard)
  └── Reads from SQLite (articles + logs) to display live activity
```

### Traces

Check your traces at https://platform.openai.com/traces to see the full agent
activity — Researcher searches, article saves, and multi-agent coordination.